In [12]:
# CELL 1
# Install all tools your AI needs
# Tap the ▶️ button to run each cell

!pip install torch numpy transformers datasets

import torch
import torch.nn as nn
import numpy as np

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Everything installed!")
print(f"   Device: {device}")
print(f"   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU")

✅ Everything installed!
   Device: cuda
   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU


In [13]:
# CELL 2
# Kaggle saves your work automatically!
# No Google Drive setup needed at all
# Your files stay safe at /kaggle/working/

import os

# This is your Kaggle save folder
# Everything here persists between sessions
SAVE = '/kaggle/working/MyGameAI'
os.makedirs(f'{SAVE}/model',       exist_ok=True)
os.makedirs(f'{SAVE}/data',        exist_ok=True)
os.makedirs(f'{SAVE}/checkpoints', exist_ok=True)

print("✅ Save folders ready!")
print(f"   Location: {SAVE}")
print("   Kaggle keeps your files automatically!")
print("   No Google Drive setup needed!")
print()
print("💡 IMPORTANT KAGGLE SETTINGS:")
print("   On the right side panel in Kaggle set these:")
print("   • Accelerator → GPU T4 x2 (free!)")
print("   • Persistence → Files (keeps your files!)")
print("   • Internet → ON (needed to install packages)")

✅ Save folders ready!
   Location: /kaggle/working/MyGameAI
   Kaggle keeps your files automatically!
   No Google Drive setup needed!

💡 IMPORTANT KAGGLE SETTINGS:
   On the right side panel in Kaggle set these:
   • Accelerator → GPU T4 x2 (free!)
   • Persistence → Files (keeps your files!)
   • Internet → ON (needed to install packages)


In [14]:
# CELL 3
# THIS IS YOUR AI BRAIN
# Read every comment carefully — it explains what each part does

import torch
import torch.nn as nn
import math
import re

# ════════════════════════════════════════
# PART A — TOKENIZER
# Converts words into numbers
# AI cannot read words — only numbers
# ════════════════════════════════════════

class SimpleTokenizer:
    """
    Turns text into numbers and back.
    """
    def __init__(self):
        self.word_to_id = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.id_to_word = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.next_id = 4

    def _tokenize(self, text):
        # CRITICAL FIX 1: This separates punctuation from words!
        # "get_tree().quit()" becomes ["get_tree", "(", ")", ".", "quit", "(", ")"]
        return re.findall(r'\w+|[^\w\s]', text.lower())

    def add_text(self, text):
        """Learn new words from text."""
        for word in self._tokenize(text):
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1

    def encode(self, text, max_length=128, pad=True):
        """Text → numbers."""
        words = self._tokenize(text)
        ids = [self.word_to_id.get(w, 3) for w in words]  # 3 = unknown word
        
        if pad:
            ids = ids[:max_length-1]  # cut if too long
            ids.append(2) # CRITICAL FIX 2: Add <END> token so AI learns to stop!
            ids += [0] * (max_length - len(ids))  # pad with zeros
        else:
            ids = ids[:max_length] # No padding for generation!
            
        return ids

    def decode(self, ids):
        """Numbers → text."""
        words = []
        for i in ids:
            if i == 2:  # END token
                break
            if i not in (0, 1):  # skip PAD and START
                words.append(self.id_to_word.get(i, "<UNK>"))
        
        # Join words and clean up spacing around punctuation
        text = " ".join(words)
        text = re.sub(r'\s+([?.!,"\'()])', r'\1', text) 
        return text

    def vocab_size(self):
        return self.next_id

    def save(self, path):
        import json
        with open(path, 'w') as f:
            json.dump({'word_to_id': self.word_to_id,
                      'id_to_word': {str(k): v for k,v in self.id_to_word.items()},
                      'next_id': self.next_id}, f)
        print(f"Tokenizer saved: {path}")

    def load(self, path):
        import json
        with open(path, 'r') as f:
            data = json.load(f)
        self.word_to_id = data['word_to_id']
        self.id_to_word = {int(k): v for k,v in data['id_to_word'].items()}
        self.next_id = data['next_id']
        print(f"Tokenizer loaded: {path}")

tokenizer = SimpleTokenizer()
print("✅ Tokenizer created!")


# ════════════════════════════════════════
# PART B — ATTENTION MECHANISM
# Helps AI focus on important words
# Like how you focus on key words when reading
# ════════════════════════════════════════

class SelfAttention(nn.Module):
    """
    Attention lets the AI focus on
    the most important words when generating an answer.

    Example:
    Question: "How do I make a JUMP in Godot?"
    AI focuses most on: JUMP and Godot
    Less on: How, do, I, make, a, in
    """

    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # These learn what to focus on
        self.query  = nn.Linear(embed_dim, embed_dim)
        self.key    = nn.Linear(embed_dim, embed_dim)
        self.value  = nn.Linear(embed_dim, embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, channels

        # Split into multiple heads
        def split_heads(tensor):
            return tensor.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.query(x))
        K = split_heads(self.key(x))
        V = split_heads(self.value(x))

        # Calculate attention scores
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        # Mask future tokens (AI should not peek at future words)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        # Convert to probabilities
        attention = torch.softmax(scores, dim=-1)

        # Apply attention to values
        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.output(out)


# ════════════════════════════════════════
# PART C — TRANSFORMER BLOCK
# One thinking layer of your AI
# Your AI has multiple of these stacked
# ════════════════════════════════════════

class TransformerBlock(nn.Module):
    """
    One layer of thinking.
    Your AI stacks multiple of these.
    More layers = smarter AI (but slower to train)
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        # Attention — focuses on important words
        self.attention = SelfAttention(embed_dim, num_heads)

        # Feed forward — processes the information
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),                      # activation function
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

        # Normalization — keeps numbers stable during training
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Attention with residual connection
        x = x + self.dropout(self.attention(self.norm1(x)))
        # Feed forward with residual connection
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x


# ════════════════════════════════════════
# PART D — YOUR COMPLETE AI MODEL
# This is the full brain
# ════════════════════════════════════════

class YourGameAI(nn.Module):
    """
    YOUR COMPLETE AI MODEL

    This is what you built from scratch.
    It takes a question as input
    and generates a game development answer.

    You can adjust these settings:
    vocab_size  = how many words it knows
    embed_dim   = how much memory per word (bigger = smarter)
    num_layers  = how many thinking layers (more = smarter but slower)
    num_heads   = how many attention heads
    max_length  = longest text it can handle
    """

    def __init__(
        self,
        vocab_size,
        embed_dim=512,    # ← increase to 512 for smarter AI
        num_layers=8,     # ← increase to 8 for smarter AI
        num_heads=8,
        ff_dim=512,
        max_length=512,
        dropout=0.1
    ):
        super().__init__()
        self.max_length = max_length

        # Word embeddings — converts word IDs to vectors
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Position embeddings — tells AI WHERE each word is
        self.pos_embed = nn.Embedding(max_length, embed_dim)

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Stack of transformer blocks (the thinking layers)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)

        # Output head — converts back to word probabilities
        self.output_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Initialize weights properly
        self._init_weights()

        # Count parameters
        params = sum(p.numel() for p in self.parameters())
        print(f"   Your AI has {params:,} parameters")

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids):
        B, T = token_ids.shape

        # Get word embeddings
        word_emb = self.word_embed(token_ids)

        # Get position embeddings
        positions = torch.arange(T, device=token_ids.device).unsqueeze(0)
        pos_emb = self.pos_embed(positions)

        # Combine word + position information
        x = self.dropout(word_emb + pos_emb)

        # Pass through all thinking layers
        for layer in self.layers:
            x = layer(x)

        # Final normalization
        x = self.norm(x)

        # Get word probabilities
        logits = self.output_head(x)
        return logits

    def generate(self, tokenizer, prompt, max_new_tokens=200, temperature=0.7, repetition_penalty=1.2):
        self.eval()
        
        # CRITICAL FIX 3: pad=False! 
        # Now the AI actually reads your prompt instead of reading zeros!
        ids = tokenizer.encode(prompt, max_length=self.max_length, pad=False)
        input_ids = torch.tensor([ids], dtype=torch.long, device=next(self.parameters()).device)

        generated = []
        with torch.no_grad():
            for _ in range(max_new_tokens):
                current = input_ids[:, -self.max_length:]
                logits = self(current)

                next_logits = logits[:, -1, :] / temperature

                if repetition_penalty != 1.0:
                    for token_id in set(input_ids[0].tolist() + generated):
                        if next_logits[0, token_id] > 0:
                            next_logits[0, token_id] /= repetition_penalty
                        else:
                            next_logits[0, token_id] *= repetition_penalty

                probs = torch.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

                if next_token.item() == 2: # Stop if END token
                    break

                generated.append(next_token.item())
                input_ids = torch.cat([input_ids, next_token], dim=1)

        return tokenizer.decode(generated)

print("✅ All AI classes defined!")
print("   Ready to create your model in the next cell")

✅ Tokenizer created!
✅ All AI classes defined!
   Ready to create your model in the next cell


In [15]:
# CELL 4 — CREATE YOUR AI MODEL
# This actually builds the brain

# You will add vocabulary from training data in Phase 3
# For now create with expanded vocab — will grow further
tokenizer = SimpleTokenizer()

# ══════════════════════════════════════════════════
# EXPANDED BASIC WORDS — v3.0
# Includes Gemini recommended bridge words
# These help AI understand English connectors
# not just Godot keywords
# ══════════════════════════════════════════════════

basic_words = """
extends node scene player enemy coin jump move speed gravity
health score level game godot gdscript characterbody2d
rigidbody2d area2d collisionshape2d sprite2d animatedsprite2d
tilemap canvaslayer label button input velocity position
signal func var const export ready process physics
android mobile touch screen apk export build
platformer shooter puzzle rpg topdown runner
design mechanic difficulty balance feel polish
audio sound music sfx animation sprite pixel

# CORE GAME DEV WORDS
extends class_name node scene player enemy npc boss
health stamina damage score level xp
checkpoint respawn save load pause menu ui hud

# GODOT 4 CORE NODES
Node Node2D CharacterBody2D StaticBody2D RigidBody2D
Area2D CollisionShape2D CollisionPolygon2D
Sprite2D AnimatedSprite2D Camera2D TileMap TileSet
CanvasLayer Control Label Button TextureButton
ProgressBar Timer AudioStreamPlayer2D AudioStreamPlayer
GPUParticles2D CPUParticles2D RayCast2D Line2D
ParallaxBackground ParallaxLayer VisibleOnScreenNotifier2D
VBoxContainer HBoxContainer MarginContainer CenterContainer
PanelContainer ScrollContainer TabContainer
TouchScreenButton VirtualKeyboard LineEdit

# GDSCRIPT FUNDAMENTALS
func var const enum static
if elif else match
for while break continue pass return
true false null
await yield signal
onready export

# INPUT AND MOVEMENT
Input InputEvent InputMap
move_and_slide move_and_collide
velocity direction delta
jump gravity acceleration friction
dash double_jump coyote_time

# MOBILE AND ANDROID
android mobile touch screen apk export
portrait landscape stretch_mode stretch_aspect
viewport scaling dpi
InputEventScreenTouch InputEventScreenDrag

# PERFORMANCE
delta fps process physics_process
queue_free object_pooling
visible_on_screen cull_mask
low_processor_mode
compression atlas batching

# GAME SYSTEMS
autoload singleton save_system load_system
game_manager state_machine fsm
scene_transition checkpoint respawn
inventory shop upgrade system

# DESIGN AND FEEL
coyote_time screen_shake hitstop
camera_zoom tween easing lerp
particles feedback sfx bgm
combo cooldown timer multiplier

# DATA TYPES AND STRUCTURES
int float bool String StringName
Array Dictionary PackedScene
Vector2 Rect2 Color Transform2D
signal Callable Resource

# BRIDGE WORDS — Gemini recommendation
# These help AI form proper English sentences
should because after before instead
attach handle control connect detect
return remove spawn reload reset
belongs requires whenever wherever
between without through during
another together separate individual
automatically immediately gradually
properly correctly accurately reliably
increase decrease subtract multiply
enable disable toggle activate
distance direction magnitude normalize
instance inherit extend override
reference pointer access retrieve
trigger emit receive respond listen
update refresh rebuild recreate
combine separate merge split filter
"""

tokenizer.add_text(basic_words)

# Create your AI
print("\n🧠 Creating YOUR AI model...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,     # memory per word
    num_layers=8,      # thinking layers
    num_heads=8,       # attention heads
    ff_dim=512,        # feedforward width — matches embed_dim
    max_length=512,    # max text length
    dropout=0.2        # prevents overfitting
)

your_ai = your_ai.to(device)
print(f"✅ YOUR AI MODEL CREATED!")
print(f"   Running on: {device}")
print(f"   Vocabulary size: {tokenizer.vocab_size()} words")
print(f"   Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")


🧠 Creating YOUR AI model...
   Your AI has 13,176,832 parameters
✅ YOUR AI MODEL CREATED!
   Running on: cuda
   Vocabulary size: 283 words
   Parameters: 13,176,832


In [16]:
import json

# This takes ALL 500+ of your examples and perfectly formats them into JSON!
with open("godot_dataset.json", "w", encoding="utf-8") as f:
    json.dump(training_data, f, indent=4)

print("✅ SUCCESS! Your JSON file has been created.")
print("Look at the right side of your screen under 'Output' -> '/kaggle/working/' to download it!")

✅ SUCCESS! Your JSON file has been created.
Look at the right side of your screen under 'Output' -> '/kaggle/working/' to download it!


In [17]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6: 100% CUSTOM GITHUB DATASET (THE PRO WAY)        ║
# ╚══════════════════════════════════════════════════════════╝
import torch
from torch.utils.data import Dataset, DataLoader
import json
import requests

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

print("⏳ 1. Downloading Master JSON Dataset from GitHub...")

# This is the RAW link to your new GitHub file
github_url = "https://raw.githubusercontent.com/Yanzkie123-567/godot-ai-android/main/godot_dataset.json"

final_training_data = []

try:
    response = requests.get(github_url)
    response.raise_for_status() 
    final_training_data = response.json()
    print(f"✅ SUCCESS: Downloaded {len(final_training_data)} real Godot examples from GitHub!")
except Exception as e:
    print(f"❌ ERROR downloading from GitHub: {e}")
    final_training_data = [{"q": "test", "a": "print('Error')"} ]

# --- TRAIN A 100% CUSTOM TOKENIZER FROM SCRATCH ---
print("⏳ 2. Building Custom Dictionary from your JSON...")
raw_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
raw_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[EOS]", "User:", "Assistant:"], vocab_size=5000)

def get_training_corpus():
    for item in final_training_data:
        yield f"User: {item['q']} Assistant: {item['a']}[EOS]"

raw_tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw_tokenizer)
tokenizer.pad_token = "[PAD]"
tokenizer.eos_token = "[EOS]"
print(f"✅ Custom Tokenizer Built! Words known: {len(tokenizer)}")

# --- NEW DATASET CLASS ---
class ProGodotDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=256):
        self.input_ids = []
        for item in data_list:
            text = f"User: {item['q']} Assistant: {item['a']}{tokenizer.eos_token}"
            encoded = tokenizer(text, truncation=True, max_length=max_length, padding="max_length")
            self.input_ids.append(encoded['input_ids'])

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.input_ids[idx])

print("⏳ 3. Encoding data into numbers...")
dataset = ProGodotDataset(final_training_data, tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# --- REBUILD THE AI BRAIN ---
print(f"🧠 Rebuilding AI...")
your_ai = YourGameAI(
    vocab_size=len(tokenizer),
    embed_dim=512,
    num_layers=6,
    num_heads=8,
    ff_dim=2048,
    max_length=256,
    dropout=0.2
).to(device)

print(f"✅ AI Rebuilt! Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")
print("🚀 READY TO TRAIN! Go run CELL 7!")

⏳ 1. Downloading Master JSON Dataset from GitHub...
✅ SUCCESS: Downloaded 564 real Godot examples from GitHub!
⏳ 2. Building Custom Dictionary from your JSON...



✅ Custom Tokenizer Built! Words known: 3151
⏳ 3. Encoding data into numbers...
🧠 Rebuilding AI...
   Your AI has 22,273,024 parameters
✅ AI Rebuilt! Parameters: 22,273,024
🚀 READY TO TRAIN! Go run CELL 7!


In [ ]:
# CELL 7 — TRAIN YOUR AI
# Improved with repetition penalty to stop word loops
# Uses User/Assistant format in test prompt

import torch.optim as optim
import time

def train_your_ai(
    model,
    dataloader,
    tokenizer,
    num_epochs=100,
    learning_rate=5e-4,
    save_dir=SAVE
):
    # Optimizer adjusts the AI weights during training
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)

    # ADVANCED AI: OneCycleLR is the industry standard for Transformers
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=learning_rate, 
        steps_per_epoch=len(dataloader), 
        epochs=num_epochs,
        pct_start=0.1
    )

    # ADVANCED AI: Label smoothing prevents the AI from becoming overconfident
    loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
    
    # ADVANCED AI: Mixed Precision Scaler (Trains 2x Faster on Kaggle GPU!)
    scaler = torch.amp.GradScaler(device)

    best_loss = float('inf')
    history = []

    print("🚀 TRAINING YOUR AI STARTED!")
    print(f"   Epochs: {num_epochs}")
    print(f"   Examples: {len(dataloader.dataset)}")
    print("=" * 50)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0
        start_time = time.time()

        for batch in dataloader:
            batch = batch.to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            optimizer.zero_grad()

            # ADVANCED AI: Autocast uses 16-bit math for lightning-fast training
            with torch.amp.autocast(device_type=device):
                logits = model(inputs)
                loss = loss_fn(
                    logits.reshape(-1, logits.size(-1)),
                    targets.reshape(-1)
                )

            # ADVANCED AI: Scaled backward pass
            scaler.scale(loss).backward()
            
            # Unscale before clipping gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            
            # --- THE FIX FOR THE RED WARNING ---
            # Only step the scheduler if the optimizer actually took a step!
            scale_before = scaler.get_scale()
            scaler.update()
            scale_after = scaler.get_scale()
            
            if scale_before <= scale_after:
                scheduler.step()
            # -----------------------------------

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        elapsed = time.time() - start_time
        history.append(avg_loss)

        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {(epoch+1):>3}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Time: {elapsed:.1f}s | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

            # Test what AI says right now using NEW format
            model.eval()
            test_answer = model.generate(
                tokenizer,
                'User: how do i make a player jump Assistant:',
                max_new_tokens=50,
                temperature=0.9,
                repetition_penalty=1.3   # ← STOPS word loops!
            )
            print(f"   AI answer preview: {test_answer[:80]}...")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'loss': best_loss,
                'vocab_size': len(tokenizer) # <--- FIXED for Hugging Face
            }, f'{save_dir}/checkpoints/best_model.pt')

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 30 == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'loss': avg_loss
            }, f'{save_dir}/checkpoints/epoch_{(epoch+1)}.pt')
            tokenizer.save_pretrained(f'{save_dir}/checkpoints') # <--- FIXED for Hugging Face
            print(f"💾 Saved checkpoint at epoch {epoch+1}")

    # Save final model
    torch.save(model.state_dict(), f'{save_dir}/model/final_model.pt')
    tokenizer.save_pretrained(f'{save_dir}/model') # <--- FIXED for Hugging Face

    print("\n" + "=" * 50)
    print("✅ TRAINING COMPLETE!")
    print(f"   Best loss: {best_loss:.4f}")
    print(f"   Final loss: {avg_loss:.4f}")
    print(f"   Lower loss = smarter AI")
    print(f"   Model saved to: {save_dir}/model/")
    return history
    
# START TRAINING YOUR AI
history = train_your_ai(
    model=your_ai,
    dataloader=dataloader,
    tokenizer=tokenizer,
    num_epochs=100,
    learning_rate=5e-4,
    save_dir=SAVE
)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║       STEP 4: CONTEXT MEMORY (CHATBOT MODE)              ║
# ╚══════════════════════════════════════════════════════════╝

# This string will hold the memory of your entire conversation
chat_memory = ""

def chat_with_godot_ai(model, tokenizer, new_question):
    global chat_memory
    model.eval()
    
    # 1. Add the new question to the memory
    chat_memory += f"User: {new_question} Assistant:"
    
    # 2. Convert the ENTIRE memory into numbers
    input_ids = tokenizer.encode(chat_memory, return_tensors="pt").to(device)
    
    # Prevent memory from getting too long and crashing the GPU
    if input_ids.shape[1] > 400:
        input_ids = input_ids[:, -400:] # Keep only the most recent conversation
    
    # 3. Generate the answer
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=150,
            temperature=0.3,
            repetition_penalty=1.2
        )
    
    # 4. Decode the answer
    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract ONLY the newest answer
    new_answer = full_output[len(chat_memory):].split("User:")[0].strip()
    
    # 5. Save the AI's answer into the memory so it remembers it for next time!
    chat_memory += f" {new_answer} \n"
    
    print(f"👤 YOU: {new_question}")
    print(f"🤖 AI:  {new_answer}")
    print("-" * 50)

# --- LET'S TEST ITS MEMORY! ---
print("Starting Chat... (Memory is empty)\n")

# Question 1
chat_with_godot_ai(your_ai, tokenizer, "how do i make a Sprite2D?")

# Question 2 (It has to remember what node we are talking about!)
chat_with_godot_ai(your_ai, tokenizer, "how do i hide it?")

# Question 3
chat_with_godot_ai(your_ai, tokenizer, "how do i delete it completely?")